# Materials 03 — Uniaxial deformation of a single crystal

We strain a ⟨100⟩ single crystal at finite temperature — in **tension and
compression** — and record the stress–strain response. Defect-free crystals
deform elastically up to a peak, then *yield* as dislocations nucleate
homogeneously; tension and compression are famously **not symmetric** in fcc
metals.

The recipe (identical to the original Tutorial 3, minus the EAM potential):
equilibrate with `fix npt`, then `fix deform` stretches the box at a constant
engineering strain rate while `fix npt` keeps the transverse directions at
zero pressure so the sample can contract freely.

In [ ]:
%pip install lammps-js matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lammps import lammps, LMP_STYLE_ATOM, LMP_TYPE_VECTOR

A0 = 1.5496          # equilibrium lattice constant from tutorial 01
TEMP = 0.1           # well below LJ melting (~0.7)
ERATE = 0.004        # engineering strain rate, 1/tau

async def deform(mode):
    sign = 1 if mode == "tension" else -1
    lmp = await lammps(output=None)
    lmp.commands_string(f"""
units lj
atom_style atomic
boundary p p p
lattice fcc {4 / A0**3:.8f}
region box block 0 6 0 6 0 6
create_box 1 box
create_atoms 1 box
mass 1 1.0
pair_style lj/cut 2.5
pair_coeff 1 1 1.0 1.0 2.5
velocity all create {TEMP} 12345 mom yes rot no
timestep 0.005
fix equil all npt temp {TEMP} {TEMP} 0.5 iso 0.0 0.0 5.0
run 2000 post no
unfix equil
fix intg all npt temp {TEMP} {TEMP} 0.5 y 0.0 0.0 5.0 z 0.0 0.0 5.0
fix pull all deform 1 x erate {sign * ERATE} units box remap x
compute csym all centro/atom fcc
""")
    lx0 = lmp.get_thermo("lx")
    strain, stress, defects = [], [], []
    for chunk in range(25):
        lmp.command("run 400 post no")
        strain.append(sign * (lmp.get_thermo("lx") - lx0) / lx0)
        stress.append(sign * -lmp.get_thermo("pxx"))   # sigma_xx, loading direction
        csym = lmp.extract_compute("csym", LMP_STYLE_ATOM, LMP_TYPE_VECTOR)
        defects.append((csym > 1.0).mean())            # stacking-fault / core signature
    return lmp, np.array(strain), np.array(stress), np.array(defects)

lmp_t, strain_t, stress_t, defects_t = await deform("tension")
lmp_c, strain_c, stress_c, defects_c = await deform("compression")
print(f"tension peak:     {stress_t.max():.2f} eps/sigma^3 at strain {strain_t[stress_t.argmax()]:.3f}")
print(f"compression peak: {stress_c.max():.2f} eps/sigma^3 at strain {strain_c[stress_c.argmax()]:.3f}")

In [ ]:
plt.figure(figsize=(5.5, 3.6))
plt.plot(strain_t, stress_t, label="tension", color="tab:red")
plt.plot(-strain_c, -stress_c, label="compression", color="tab:blue")
plt.axhline(0, lw=0.5, color="gray"); plt.axvline(0, lw=0.5, color="gray")
plt.xlabel("engineering strain"); plt.ylabel("stress σₓₓ (ε/σ³)")
plt.legend(); plt.tight_layout(); plt.show()

Both curves rise linearly (elastic regime — same slope, the Young's
modulus), then drop sharply when **dislocations nucleate** and the crystal
yields. The peak stresses differ between tension and compression — the
tension–compression asymmetry of dislocation nucleation in fcc metals. These
ideal strengths are very high because a *perfect* crystal has no defects to
start plasticity; real crystals yield far earlier.

## Watch the defects appear

The **centrosymmetry parameter** is ≈ 0 for atoms in a perfect fcc
environment (it is insensitive to elastic strain!) and jumps inside
dislocation cores and stacking faults. We recorded, at every chunk, the
fraction of atoms with centrosymmetry > 1 — plot it against the stress and
the mechanism is unmistakable:

In [ ]:
fig, ax1 = plt.subplots(figsize=(5.8, 3.6))
ax1.plot(strain_t, stress_t, color="tab:red", label="stress")
ax1.set_xlabel("engineering strain (tension)")
ax1.set_ylabel("stress σₓₓ (ε/σ³)", color="tab:red")
ax2 = ax1.twinx()
ax2.plot(strain_t, 100 * defects_t, color="tab:gray", ls="--", label="defect atoms")
ax2.set_ylabel("atoms with centrosymmetry > 1 (%)", color="tab:gray")
fig.tight_layout(); plt.show()

print(f"peak defect fraction: {defects_t.max():.1%} "
      f"(at strain {strain_t[defects_t.argmax()]:.3f})")
lmp_t.close(); lmp_c.close()

The crystal is defect-free through the *entire* elastic rise — then, at the
exact strain where the stress collapses, hundreds of atoms light up at once:
**homogeneous dislocation nucleation** (the debris is stacking faults left by
partial dislocations). In a box this small the dislocations can sweep out
through the periodic boundaries and the crystal may even re-anneal and load
up again — enlarge the box to keep more debris around.

**Exercises**
- Lower `ERATE` or enlarge the box: the yield stress drops. Why?
- Rerun at `TEMP = 0.3`. Thermal fluctuations help nucleation — the peak
  moves down.
- Plot the *transverse* box length vs. strain and estimate Poisson's ratio.

Next: [04 — Building a grain boundary](04-grain-boundary.ipynb): real
materials are not single crystals.